# Credit-Constrained Model: When Targeting Can Actually Help

**Buera, Kaboski & Shin (2011)-style collateral constraint with heterogeneous net worth**

The base model showed that targeting productive firms still lowers output (vs. doing nothing).
This extension adds a **real, pre-existing friction**: the collateral constraint k ≤ λ·a, where
wealth and talent are uncorrelated. Now talented entrants can be stuck below efficient scale
simply because they're poor — exactly the kind of friction that targeted policy can correct.

This notebook solves four economies and compares aggregate output:
1. **Frictionless**: No capital constraint (upper-bound benchmark)
2. **Constrained**: Finite collateral multiplier, no policy
3. **Targeted**: Credit access favors top-quintile entrants
4. **Mistargeted**: Credit access favors bottom-quintile entrants

## 1. Setup

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# Add src to path (go up one directory from notebooks/)
sys.path.insert(0, '../src')

from ge_model import (
    run_credit_scenario,
    CALIBRATION_CREDIT,
    s_grid_credit,
    plot_credit_output,
    plot_credit_constrained_share,
)

print("[OK] Imports successful")
print(f"[OK] Working directory: {os.getcwd()}")

## 2. Calibration Parameters

In [ ]:
# Display calibration
calib_display = {
    'Parameter': [
        'Productivity grid points',
        'AR(1) persistence (ρ)',
        'Innovation std dev (σ)',
        'Capital share (θ_k)',
        'Labor share (θ_n)',
        'Returns to scale (θ_k + θ_n)',
        'Capital rental rate (r)',
        'Discount factor (β)',
        'Fixed operating cost',
        'Entry cost',
        'Aggregate labor supply',
        'Base collateral multiplier (λ_base)',
        'Boosted multiplier (λ_boost)',
    ],
    'Value': [
        CALIBRATION_CREDIT['N_Z'],
        CALIBRATION_CREDIT['RHO'],
        CALIBRATION_CREDIT['SIGMA'],
        CALIBRATION_CREDIT['THETA_K'],
        CALIBRATION_CREDIT['THETA_N'],
        f"{CALIBRATION_CREDIT['THETA_K'] + CALIBRATION_CREDIT['THETA_N']:.2f}",
        CALIBRATION_CREDIT['R_RATE'],
        CALIBRATION_CREDIT['BETA'],
        f"{CALIBRATION_CREDIT['CF']:.2f}",
        f"{CALIBRATION_CREDIT['CE']:.2f}",
        CALIBRATION_CREDIT['L_SUPPLY'],
        CALIBRATION_CREDIT['LAMBDA_BASE'],
        CALIBRATION_CREDIT['LAMBDA_BOOST'],
    ]
}
calib_df = pd.DataFrame(calib_display)
print("Credit Model Calibration (stylized, illustrative):")
print(calib_df.to_string(index=False))
print(f"\nNet worth types (a): {CALIBRATION_CREDIT['A_VALS']}")
print("  Note: a is drawn independently of productivity (BKS point: wealth ≠ talent)")

## 3. Model Specification: Two Inputs + Collateral Constraint

In [ ]:
model_spec = """
FIRM PROBLEM (two inputs: capital k, labor n):
  max_{k,n}  p · s · k^θ_k · n^θ_n - r·k - w·n
  
  subject to:  k ≤ λ · a   (collateral constraint)
  
  where:
    • s = productivity (AR(1) in logs, independent of wealth a)
    • a = net worth / collateral capacity (drawn at entry, fixed thereafter)
    • λ = collateral multiplier (policy can vary this)
    • θ_k = 0.30 (capital share)
    • θ_n = 0.35 (labor share)
    • r = 0.10 (capital rental rate, interest + depreciation)
    • w = 1.0 (wage)
  
  Solution method:
    • Closed-form unconstrained solution (two-input Cobb-Douglas)
    • Check if k_unconstrained > λ·a
    • If yes: pin k = λ·a, re-solve for n from FOC

VALUE FUNCTION:
  V(s|a,λ) = π(s, k(s), n(s)) - cf + β·E[max(V(s'|a,λ), 0)]
  
  Exit rule: firm exits if V < 0
  Constraint binding? Tracked per firm

KEY INSIGHT (Buera, Kaboski & Shin):
  Because productivity s and wealth a are uncorrelated at entry, some high-s
  (talented) firms are constrained while some low-s (weak) firms are not.
  This creates a *real* friction, not just a wedge.
  → Relaxing λ for talented firms closes a genuine inefficiency.
  → Relaxing λ for weak firms does nothing (they don't want more capital).
"""

print(model_spec)

## 4. Solve Four Regimes

In [ ]:
# Run all four regimes
regimes_credit = ["frictionless", "constrained", "targeted", "mistargeted"]
results_credit = {}

for regime in regimes_credit:
    print(f"Solving {regime}...", end=" ")
    res = run_credit_scenario(regime)
    results_credit[regime] = res
    print(f"✓ p*={res['p']:.4f}, Y={res['aggregate_output']:.4f}, constrained={100*res['constrained_share']:.1f}%")

print("\n✓ All regimes solved.")

## 5. Results Table

In [ ]:
# Build results table
rows = []
Y_frictionless = results_credit["frictionless"]["aggregate_output"]
Y_constrained = results_credit["constrained"]["aggregate_output"]

for regime in regimes_credit:
    res = results_credit[regime]
    gap_closed = 100 * (res["aggregate_output"] - Y_constrained) / (Y_frictionless - Y_constrained)
    
    rows.append({
        "Regime": regime.capitalize(),
        "Eq. Price (p*)": res["p"],
        "Aggregate Output": res["aggregate_output"],
        "Output (% of frictionless)": 100 * res["aggregate_output"] / Y_frictionless,
        "% of friction gap closed": gap_closed,
        "Exit Rate": res["exit_rate"],
        "Constrained Share (%)": 100 * res["constrained_share"],
    })

table_credit = pd.DataFrame(rows)
print("\nEQUILIBRIUM RESULTS:")
print(table_credit.to_string(index=False))

# Save to CSV (go up one directory)
os.makedirs('../results/credit', exist_ok=True)
table_credit.to_csv('../results/credit/results_table.csv', index=False, float_format='%.4f')
print("\n[OK] Saved: ../results/credit/results_table.csv")

## 6. Visualizations

In [ ]:
# Chart 1: Output Levels
labels_credit = {
    "frictionless": "Frictionless\n(no credit constraint)",
    "constrained": "Constrained, no policy\n(wealth uncorrelated w/ talent)",
    "targeted": "Targeted credit access\n(top-quintile entrants favored)",
    "mistargeted": "Mistargeted credit access\n(bottom-quintile entrants favored)",
}

fig, ax = plot_credit_output(results_credit, regimes_credit, labels_credit, save_path='../results/credit/output.png')
plt.show()
print("[OK] Saved: ../results/credit/output.png")

In [ ]:
# Chart 2: Constrained Share
fig, ax = plot_credit_constrained_share(results_credit, regimes_credit, labels_credit, save_path='../results/credit/constrained_share.png')
plt.show()
print("[OK] Saved: ../results/credit/constrained_share.png")

## 7. Key Findings

In [ ]:
finding1 = f"""
FINDING 1: With a real friction, targeting productive firms WORKS

  Frictionless output:              {Y_frictionless:.4f}
  Constrained output (no policy):   {Y_constrained:.4f}
  Targeted policy output:           {results_credit['targeted']['aggregate_output']:.4f}
  
  Friction gap closed by targeting: {100 * (results_credit['targeted']['aggregate_output'] - Y_constrained) / (Y_frictionless - Y_constrained):.1f}%
  
  → Targeted credit access recovers about half the loss from the constraint.
  → This is fundamentally different from the base model, where targeting always hurts.
"""

finding2 = f"""
FINDING 2: Mistargeting here is WASTED, not HARMFUL

  Mistargeted policy output:  {results_credit['mistargeted']['aggregate_output']:.4f}
  Constrained output:         {Y_constrained:.4f}
  
  Why the difference from the base model?
    • Base model: tax/subsidy forces distortion (harmful even on strong firms)
    • Credit model: relaxing collateral is just an *option* (weak firms don't use it)
  
  → You can't force a firm to borrow for capital it has no productive use for.
  → Mistargeting wastes the policy but doesn't destroy output.
"""

finding3 = f"""
FINDING 3: The central lesson

  Whether "pick the winners" is good or bad policy depends entirely on
  whether there's a real, independent market failure the policy can correct.
  
  • In a frictionless model: targeting is ALWAYS bad (misallocates perfect equilibrium)
  • In a model with genuine financing friction: targeting can be EXACTLY right
    (provided the targeting is accurate, which is the hard part in practice)
"""

print(finding1)
print(finding2)
print(finding3)

In [ ]:
# Constraint intensity by regime
print("\nFRICTION INTENSITY (constrained share):")
for regime in regimes_credit:
    share = 100 * results_credit[regime]['constrained_share']
    print(f"  {regime:15s}: {share:6.1f}% of firms at capital limit")